In [49]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


In [50]:
pooled = pd.read_csv("../data/pooled_estimates/pooled_full_specification.csv")
ASCFR_PATH = "../data/raw/expenditure/ASC-FR CSV 2023-24 (Values) V2.csv"
SALT_PATH = ("../data/raw/expenditure/ASCFR and SALT Data Tables 2023-24.xlsx")
IMD_PATH = "../data/raw/deprivation/File_11_-_IoD2025_Local_Authority_District_Summaries__upper-tier__v2.xlsx"
RUC_PATH = "../data/raw/census/urban_rural_classification.csv"


In [51]:
# Exclude local auhtorities with low population
SMALL    = ["E09000001", "E06000053"]     # City of London, Isles of Scilly

SALT (Number of ASC clients)

In [52]:
# Definitions

SALT_SHEET = "T33"

CASSR_PREFIXES = ("E06", "E08", "E09", "E10")
CASSR_PATTERN = r"^E(06|08|09|10)\d{6}$"

BASES = {
    "inyear_65plus": "in-year",
    "yearend_65plus": "year-end",
    "m12_65plus": "12-month",
}

PRIMARY_BASIS = "inyear_65plus"

EXPECTED_COLUMNS = [
    "area_code",      
    "area_num",       
    "area_name",     
    "region_code", "region_name",
    "inyear_18_64", "inyear_65plus", "inyear_total",
    "yearend_18_64", "yearend_65plus", "yearend_total",
    "m12_18_64", "m12_65plus", "m12_total",
]



In [53]:
salt_raw = pd.read_excel(SALT_PATH, sheet_name=SALT_SHEET, header=None)
print(f"  sheet read: {salt_raw.shape[0]} rows x {salt_raw.shape[1]} columns")
code_matches = {
    c: salt_raw[c].astype(str).str.strip().str.match(CASSR_PATTERN, na=False).sum()
    for c in salt_raw.columns
}
code_col = max(code_matches, key=code_matches.get)
if code_matches[code_col] == 0:
    raise ValueError(
        f"no column in {SALT_SHEET} contains CASSR codes matching {CASSR_PATTERN}"
    )

is_cassr = salt_raw[code_col].astype(str).str.strip().str.match(CASSR_PATTERN, na=False)
data_rows = salt_raw.index[is_cassr]

  sheet read: 175 rows x 15 columns


In [54]:
block

,area_code,area_num,area_name,region_code,region_name,inyear_18_64,inyear_65plus,inyear_total,yearend_18_64,yearend_65plus,yearend_total,m12_18_64,m12_65plus,m12_total
20,E06000057,104,Northumberland,E12000001,North East,2150.0,5275.0,7425.0,1835.0,3350.0,5185.0,1590.0,2305.0,3895.0
21,E08000037,106,Gateshead,E12000001,North East,885.0,2865.0,3750.0,790.0,1895.0,2685.0,710.0,1370.0,2080.0
22,E08000021,107,Newcastle upon Tyne,E12000001,North East,1695.0,3150.0,4845.0,1510.0,2265.0,3775.0,1260.0,1565.0,2820.0
23,E08000022,108,North Tyneside,E12000001,North East,1180.0,3010.0,4185.0,1070.0,2170.0,3245.0,865.0,1210.0,2075.0
24,E08000023,109,South Tyneside,E12000001,North East,850.0,2900.0,3750.0,685.0,1690.0,2375.0,545.0,935.0,1480.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,E06000024,910,North Somerset,E12000009,South West,1085.0,2110.0,3195.0,975.0,1515.0,2490.0,800.0,920.0,1720.0
169,E06000025,911,South Gloucestershire,E12000009,South West,1685.0,2460.0,4140.0,1535.0,1705.0,3240.0,1300.0,1095.0,2400.0
170,E10000008,912,Devon,E12000009,South West,5020.0,8700.0,13720.0,4470.0,6035.0,10505.0,3960.0,4140.0,8100.0
171,E06000026,913,Plymouth,E12000009,South West,1705.0,2625.0,4330.0,1545.0,1820.0,3370.0,1365.0,1195.0,2560.0


In [55]:
block = salt_raw.loc[data_rows, code_col:].copy()
n_extra = block.shape[1] - len(EXPECTED_COLUMNS)
block.columns = EXPECTED_COLUMNS + [f"unnamed_{i + 1}" for i in range(n_extra)]

count_cols = [c for c in EXPECTED_COLUMNS
              if c.startswith(("inyear", "yearend", "m12"))]
for c in count_cols:
    block[c] = pd.to_numeric(
        block[c].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )

print(f"  authorities in block: {len(block)}")
for basis, label in BASES.items():
    n = int(block[basis].notna().sum())
    print(f"  {label:<9} basis: {n} of {len(block)} with a numeric 65+ count")

for basis, label in BASES.items():
    absent = block.loc[block[basis].isna(), "area_name"].astype(str)


for basis in ("inyear", "yearend", "m12"):
    parts = block[f"{basis}_18_64"] + block[f"{basis}_65plus"]
    discrepancy = (parts - block[f"{basis}_total"]).abs()
    n_bad = int((discrepancy > 5).sum())



salt = block[["area_code", "area_name"] + count_cols].copy()
salt["area_code"] = salt["area_code"].astype(str).str.strip()

  authorities in block: 153
  in-year   basis: 152 of 153 with a numeric 65+ count
  year-end  basis: 152 of 153 with a numeric 65+ count
  12-month  basis: 151 of 153 with a numeric 65+ count


In [56]:
salt

,area_code,area_name,inyear_18_64,inyear_65plus,inyear_total,yearend_18_64,yearend_65plus,yearend_total,m12_18_64,m12_65plus,m12_total
20,E06000057,Northumberland,2150.0,5275.0,7425.0,1835.0,3350.0,5185.0,1590.0,2305.0,3895.0
21,E08000037,Gateshead,885.0,2865.0,3750.0,790.0,1895.0,2685.0,710.0,1370.0,2080.0
22,E08000021,Newcastle upon Tyne,1695.0,3150.0,4845.0,1510.0,2265.0,3775.0,1260.0,1565.0,2820.0
23,E08000022,North Tyneside,1180.0,3010.0,4185.0,1070.0,2170.0,3245.0,865.0,1210.0,2075.0
24,E08000023,South Tyneside,850.0,2900.0,3750.0,685.0,1690.0,2375.0,545.0,935.0,1480.0
...,...,...,...,...,...,...,...,...,...,...,...
168,E06000024,North Somerset,1085.0,2110.0,3195.0,975.0,1515.0,2490.0,800.0,920.0,1720.0
169,E06000025,South Gloucestershire,1685.0,2460.0,4140.0,1535.0,1705.0,3240.0,1300.0,1095.0,2400.0
170,E10000008,Devon,5020.0,8700.0,13720.0,4470.0,6035.0,10505.0,3960.0,4140.0,8100.0
171,E06000026,Plymouth,1705.0,2625.0,4330.0,1545.0,1820.0,3370.0,1365.0,1195.0,2560.0


Expenditure

In [57]:
# Definitions

MEASURES = ["gross_65"]
PRIMARY  = ["gross_65"]
COVS     = ["idaopi_score", "pct_rural_oa", "inst_share_65", "income_share_65"]

# Tables to be used for the analysis
UUIDS = {
    "gross_65_physical": "8320215", "net_65_physical": "8320214",
    "gross_65_sensory":  "8320415", "net_65_sensory":  "8320414",
    "gross_65_memcog":   "8320615", "net_65_memcog":   "8320614",
    "gross_65_learndis": "8320815", "net_65_learndis": "8320814",
    "gross_65_mentalh":  "8321015", "net_65_mentalh":  "8321014",
    "gross_all":         "8325115", "net_all":         "8325114",
    "gross_65lts_nursing":     "8128115",
    "gross_65lts_residential": "8128215",
    "gross_65lts_total":       "8128915",
}
PSR = ["physical", "sensory", "memcog", "learndis", "mentalh"]


In [58]:
ascfr_raw = pd.read_csv(ASCFR_PATH, dtype=str, low_memory=False)
ascfr_raw.columns = [c.strip().upper() for c in ascfr_raw.columns]
print(f"  rows: {len(ascfr_raw):,}")


  rows: 509,375


In [59]:
ascfr = ascfr_raw[
    ascfr_raw["GEOGRAPHY_LEVEL"].str.strip().str.lower() == "local authority"
].copy()
print(f"  rows at local-authority level: {len(ascfr):,}")

  rows at local-authority level: 478,125


In [60]:
ascfr["UUID"] = ascfr["UUID"].astype(str).str.strip()
ascfr["ITEMVALUE"] = pd.to_numeric(
    ascfr["ITEMVALUE"].astype(str).str.replace(",", "", regex=False).str.strip(),
    errors="coerce",
)

In [61]:
items = ascfr[ascfr["UUID"].isin(UUIDS.values())].copy()
print(f"  rows matching the item list: {len(items):,}")
items["measure"] = items["UUID"].map({v: k for k, v in UUIDS.items()})

  rows matching the item list: 2,295


In [62]:
asc = (
    items.pivot_table(
        index=["GEOGRAPHY_CODE", "DH_GEOGRAPHY_NAME",
               "REGION_GO_CODE", "GEOGRAPHY_NAME"],
        columns="measure",
        values="ITEMVALUE",
        aggfunc="sum",
    )
    .reset_index()
    .rename(columns={
        "GEOGRAPHY_CODE": "area_code",
        "DH_GEOGRAPHY_NAME": "area_name",
        "REGION_GO_CODE": "region_code",
        "GEOGRAPHY_NAME": "region_name",
    })
)
asc.columns.name = None

In [63]:
money_cols = [c for c in asc.columns if c.startswith(("gross_", "net_"))]
asc[money_cols] = asc[money_cols] * 1_000.0

In [64]:
all_zero = asc[(asc[money_cols].fillna(0) == 0).all(axis=1)]
if len(all_zero):
    print(f"  {len(all_zero)} authority/authorities reporting zero")
    print(all_zero[["area_code", "area_name"]].to_string(index=False))
    asc.loc[all_zero.index, money_cols] = np.nan
 

  1 authority/authorities reporting zero
area_code               area_name
E06000064 Westmorland and Furness


In [65]:
asc["gross_65"] = asc[[f"gross_65_{p}" for p in PSR]].sum(axis=1, min_count=1)


In [66]:
asc["inst_share_65"] = (
    asc[["gross_65lts_nursing", "gross_65lts_residential"]].sum(axis=1, min_count=1)
    / asc["gross_65lts_total"]
)

asc.head()

,area_code,area_name,region_code,region_name,gross_65_learndis,gross_65_memcog,gross_65_mentalh,gross_65_physical,gross_65_sensory,gross_65lts_nursing,...,gross_65lts_total,gross_all,net_65_learndis,net_65_memcog,net_65_mentalh,net_65_physical,net_65_sensory,net_all,gross_65,inst_share_65
0,E06000001,Hartlepool,E12000001,North East,2076000.0,8181000.0,1550000.0,14773000.0,26000.0,3388000.0,...,24503000.0,49843000.0,1256000.0,6739000.0,1106000.0,10894000.0,26000.0,42049000.0,26606000.0,0.686814
1,E06000002,Middlesbrough,E12000001,North East,3450000.0,5193000.0,4186000.0,20783000.0,143000.0,2617000.0,...,31680000.0,81590000.0,2975000.0,3187000.0,3209000.0,14843000.0,118000.0,67891000.0,33755000.0,0.693718
2,E06000003,Redcar and Cleveland,E12000001,North East,1100000.0,4497000.0,1754000.0,28020000.0,122000.0,3496000.0,...,32052000.0,69374000.0,965000.0,3012000.0,1148000.0,19137000.0,65000.0,55594000.0,35493000.0,0.658586
3,E06000004,Stockton-on-Tees,E12000001,North East,3843000.0,8749000.0,2557000.0,26259000.0,533000.0,4845000.0,...,37495000.0,87571000.0,3385000.0,5374000.0,2000000.0,16178000.0,340000.0,69276000.0,41941000.0,0.716709
4,E06000005,Darlington,E12000001,North East,2725000.0,3408000.0,2207000.0,16031000.0,63000.0,2006000.0,...,20431000.0,55926000.0,2360000.0,1674000.0,1686000.0,9290000.0,53000.0,44619000.0,24434000.0,0.714796


Care "care_1000_est" estimates vs provision

In [67]:
mod = pooled.merge(asc, left_on="area_code", right_on = "area_code", how="outer")
mod = mod.merge(salt, on="area_code", how="left")

In [68]:
for m in MEASURES:
    mod[f"{m}_per_65plus"] = mod[m] / mod["population_65plus"]
 
for basis, label in BASES.items():
    mod[f"clients_per_1000_{basis.replace('_65plus', '')}"] = (
        mod[basis] / mod["population_65plus"] * 1000
    )

In [69]:
excluded = mod.loc[mod["area_code"].isin(SMALL), ["area_code", "area_name"]]
m = mod[~mod["area_code"].isin(SMALL)].copy()
print(f"excluded {len(excluded)} of {len(mod)}:")
print(excluded.to_string(index=False))
print(f"retained: {len(m)}")

excluded 2 of 153:
area_code       area_name
E06000053 Isles of Scilly
E09000001  City of London
retained: 151


In [82]:
N_LABEL = 5

OUTCOMES = [
    ("gross_65_per_65plus",     "Gross current expenditure per resident 65+"),
    ("clients_per_1000_inyear", "Clients per 1,000 aged 65+"),
]

for col, label in OUTCOMES:
    sub = m.dropna(subset=["care_1000_est", col]).copy()
    fit = sm.OLS(sub[col], sm.add_constant(sub[["care_1000_est"]])).fit()

    m.loc[sub.index, f"fitted_{col}"]  = fit.fittedvalues
    m.loc[sub.index, f"resid_{col}"]   = fit.resid
    m.loc[sub.index, f"zresid_{col}"]  = fit.resid / fit.resid.std(ddof=1)

    table = pd.DataFrame({
        "area_name": sub["area_name"],
        "region": sub["region_name"],
        "care_1000_est": sub["care_1000_est"].round(1),
        "observed": sub[col].round(1),
        "fitted": fit.fittedvalues.round(1),
        "residual": fit.resid.round(1),
        "z": (fit.resid / fit.resid.std(ddof=1)).round(2),
    }).sort_values("residual")

    print(f"{label}    n = {int(fit.nobs)}    "
          f"residual SD = {fit.resid.std(ddof=1):.1f}")
    print(f"\n{N_LABEL} largest negative residuals (provision below fitted level):")
    print(table.head(N_LABEL).to_string(index=False))
    print(f"\n{N_LABEL} largest positive residuals (provision above fitted level):")
    print(table.tail(N_LABEL).iloc[::-1].to_string(index=False))
    print()

for col, label in OUTCOMES:
    sub = m.dropna(subset=["care_1000_est", col])
    r = sub["care_1000_est"].corr(sub[col])
    print(f"  {label:<44} r = {r:.3f}   R2 = {r ** 2:.3f}   n = {len(sub)}")

Gross current expenditure per resident 65+    n = 150    residual SD = 321.5

5 largest negative residuals (provision below fitted level):
              area_name                   region  care_1000_est  observed  fitted  residual     z
               Barnsley Yorkshire and The Humber          420.5     928.8  1487.5    -558.6 -1.74
           Lincolnshire            East Midlands          377.6     726.3  1191.4    -465.0 -1.45
North East Lincolnshire Yorkshire and The Humber          399.7     880.9  1343.9    -462.9 -1.44
              Rotherham Yorkshire and The Humber          412.1     974.5  1429.5    -455.1 -1.42
                  Derby            East Midlands          408.4     958.4  1403.6    -445.3 -1.39

5 largest positive residuals (provision above fitted level):
             area_name region  care_1000_est  observed  fitted  residual    z
         Tower Hamlets London          459.0    3889.8  1752.9    2136.9 6.65
               Hackney London          418.4    2442.6 

Coefficients of variation

In [74]:
x = mod.loc[~mod["area_code"].isin(["E09000001", "E06000053"]),
            "clients_per_1000_inyear"].dropna().to_numpy(dtype=float)

cv = x.std(ddof=1) / x.mean()

rng = np.random.default_rng(2024)
samples = x[rng.integers(0, len(x), size=(10_000, len(x)))]
boot = samples.std(axis=1, ddof=1) / samples.mean(axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"clients per 1,000 aged 65+:  mean {x.mean():.1f}   sd {x.std(ddof=1):.1f}   "
      f"CV {cv:.3f}  [{lo:.3f}, {hi:.3f}]   n = {len(x)}")

clients per 1,000 aged 65+:  mean 61.0   sd 18.5   CV 0.304  [0.272, 0.332]   n = 150


In [92]:
x = mod.loc[~mod["area_code"].isin(["E09000001", "E06000053"]),
            "gross_65_per_65plus"].dropna().to_numpy(dtype=float)

cv = x.std(ddof=1) / x.mean()

rng = np.random.default_rng(2024)
samples = x[rng.integers(0, len(x), size=(10_000, len(x)))]
boot = samples.std(axis=1, ddof=1) / samples.mean(axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"gross expenditure per resident 65+:  mean {x.mean():.1f}   "
      f"sd {x.std(ddof=1):.1f}   CV {cv:.3f}  [{lo:.3f}, {hi:.3f}]   n = {len(x)}")

gross expenditure per resident 65+:  mean 1248.8   sd 372.6   CV 0.298  [0.220, 0.382]   n = 150


Deprivation

In [102]:
iod = pd.read_excel(IMD_PATH)

In [97]:
DEPRIVATION_SHEETS = {
    "IMD":        ("imd",        ["IMD25", "IMD"]),
    "Income":     ("income",     ["Income"]),
    "Employment": ("employment", ["Employment"]),
    "Education":  ("education",  ["Education, Skills and Training"]),
    "Health":     ("hdd",        ["Health Deprivation and Disability"]),
    "Crime":      ("crime",      ["Crime"]),
    "Barriers":   ("barriers",   ["Barriers to Housing and Services"]),
    "Living":     ("living_env", ["Living Environment"]),
    "IDACI":      ("idaci",      ["IDACI"]),
    "IDAOPI":     ("idaopi",     ["IDAOPI"]),
}

DEPRIVATION_MEASURES = {
    "average score": "score",
    "average rank": "avg_rank",
    "proportion of lsoas in most deprived 10% nationally": "prop10",
    "extent": "extent",
    "local concentration": "local_conc",
    "scale": "scale",
}
 

In [98]:
actual_sheets = {str(s).strip().lower(): s for s in iod.sheet_names}
resolved_sheets = {e: actual_sheets[e.strip().lower()]
                   for e in DEPRIVATION_SHEETS
                   if e.strip().lower() in actual_sheets}

if len(resolved_sheets) < len(DEPRIVATION_SHEETS):
    absent = [s for s in DEPRIVATION_SHEETS if s not in resolved_sheets]
    raise KeyError(
        f"deprivation domains not found in {IMD_PATH}: {absent}. "
        f"Sheets present: {iod.sheet_names}. The published IoD 2025 "
        "upper-tier summary file (File 11) carries one sheet per domain; a file "
        "missing these is a different release or a partial extract."
    )

deprivation = None

for sheet_name, (prefix, labels) in DEPRIVATION_SHEETS.items():
    sheet = iod.parse(resolved_sheets[sheet_name])


    sheet.columns = [str(c).strip() for c in sheet.columns]


    code_cols = [c for c in sheet.columns
                 if c.startswith("Upper Tier Local Authority District code")]
    if not code_cols:
        raise KeyError(f"no authority code column on sheet '{sheet_name}'")

    selected = {code_cols[0]: "area_code"}
    for col in sheet.columns:
        for label in labels:
            if not col.lower().startswith(label.lower()):
                continue
            remainder = col[len(label):].lstrip(" -").strip().lower()
            if remainder in DEPRIVATION_MEASURES:
                selected[col] = f"{prefix}_{DEPRIVATION_MEASURES[remainder]}"
            break 

    domain = sheet[list(selected)].rename(columns=selected)

    if domain["area_code"].duplicated().any():
        raise ValueError(f"duplicate authority codes on sheet '{sheet_name}'")

    print(f"  {sheet_name:<11} {len(domain):>4} rows  "
          f"{[c for c in domain.columns if c != 'area_code']}")

    deprivation = domain if deprivation is None else deprivation.merge(
        domain, on="area_code", how="outer")


score_cols = [c for c in deprivation.columns if c.endswith("_score")]

deprivation.head()

AttributeError: 'DataFrame' object has no attribute 'sheet_names'

In [99]:
for label, source in [("deprivation", deprivation), ("rural-urban", ruc)]:
    before = len(mod)
    mod = mod.merge(source, on="area_code", how="left", indicator="_match")

    if len(mod) != before:
        raise ValueError(
            f"{label} merge changed the row count from {before} to {len(mod)}"
        )

    unmatched = mod.loc[mod["_match"] == "left_only", ["area_code", "area_name"]]
    print(f"  {label:<12} matched {int((mod['_match'] == 'both').sum()):>3} "
          f"of {len(mod)} authorities")
    if len(unmatched):
        print(f"    authorities with no {label} record:")
        print(unmatched.to_string(index=False))

    mod = mod.drop(columns="_match")

NameError: name 'deprivation' is not defined

Rural/Urban

In [100]:
ruc_raw = pd.read_csv(RUC_PATH)
ruc = ruc_raw[ruc_raw["UTLA23CD"].str.startswith("E", na=False)].rename(columns={
    "UTLA23CD": "area_code",
    "Rural_Urban_flag": "rural_urban_flag",
    "RUC21_settlement_class": "settlement_class",
    "Proportion_of_population_in_rural_OAs____": "pct_rural_oa",
    "Proportion_of_population_in_OAs_further_from_a_major_town_or_city____":
        "pct_remote_oa",
})[["area_code", "rural_urban_flag", "settlement_class",
    "pct_rural_oa", "pct_remote_oa"]]